In [9]:
# this script is for get proper orthogroups and use meta-gene method to get individual atlas, 
# then merge into a vertebrate-level orthogroup level atlas for plotting
suppressPackageStartupMessages({
    require(Seurat)
    require(dplyr)
})

In [6]:
orthogroups <- read.delim('/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/02.gene_relationships/run5/results/Ortho_pipeline/OrthoFinder/Orthogroups/Orthogroups.tsv')
# at least one copy for 4 species
orthogroups <- orthogroups %>% select(c('Orthogroup', 'Bflo', 'Pmar', 'Pvit', 'Mmus', 'Hsap'))  %>% 
    filter(Bflo != '' & Pmar != '' & Pvit != '' & Mmus != '' & Hsap != '')

In [7]:
# calculate number of genes by calcualting commas in it
count_commas <- function(x) {
  sapply(gregexpr(",", x), function(match) ifelse(match[1] == -1, 0, length(match)))
}
number_genes <- data.frame(apply(orthogroups, c(1,2), count_commas))

# retain orthogroups with 5 copies max for each four species
orthogroups <- orthogroups[which(number_genes$Bflo <= 4 & number_genes$Pmar <= 4 & number_genes$Pvit <= 4 & number_genes$Mmus <= 4 & number_genes$Hsap <= 4), ]
orthogroups$Bflo <- gsub('_', '-', orthogroups$Bflo)

In [4]:
path <- '/mnt/data01/yuanzhen/01.Vertebrate_cell_evo/01.data/02.atlas_final/2.samap/4.final/'

In [5]:
meta <- c('DonorID','Refined family', 'Refined subtype', 'Species')
get_metagene_obj <- function(obj, orthogroups, species){
    
    test <- orthogroups %>% dplyr::select(c(Orthogroup, species)) %>% 
        tidyr::separate_rows(species, sep = ",\\s*") %>% as.data.frame()
    test <- test[test[,species] %in% rownames(obj), ]
    
    raw <- GetAssayData(obj, layer = "counts")
    meta_raw <- Matrix.utils::aggregate.Matrix(
        raw[test[,species], ],
        groupings = test$Orthogroup,
        fun = "sum"
    )
    seurat_object <- CreateSeuratObject(counts = meta_raw, meta.data = obj@meta.data[,meta])
    seurat_object <- SCTransform(seurat_object, variable.features.n = 3000, verbose = FALSE)
    return(seurat_object)
}

In [11]:
Pmar <- readRDS(paste0(path, 'Pmar.non_neurons.iter_cluster_annotated.rds'))
Pmar <- subset(Pmar, cells = rownames(Pmar@meta.data)[which(Pmar@meta.data$`Refined family` %in% 
    c('Ependymal cells', 'Astrocytes', 'Oligodendrocytes', 'Oligodendrocyte precursor cells',
     'Vascular cells', 'Microglia', 'Fibroblasts'))])
Pmar_metagene <- get_metagene_obj(Pmar, orthogroups, "Pmar")
rm(Pmar)
saveRDS(Pmar_metagene, "Pmar.glia.metagene.rds")

In [12]:
Pvit <- readRDS(paste0(path, 'Pvit.non_neurons.iter_cluster_annotated.rds'))
Pvit <- subset(Pvit, cells = rownames(Pvit@meta.data)[which(Pvit@meta.data$`Refined family` %in% 
    c('Ependymal cells', 'Astrocytes', 'Oligodendrocytes', 'Oligodendrocyte precursor cells',
     'Vascular cells', 'Microglia', 'Fibroblasts'))])
Pvit_metagene <- get_metagene_obj(Pvit, orthogroups, "Pvit")
rm(Pvit)
saveRDS(Pvit_metagene, "Pvit.glia.metagene.rds")

In [13]:
Mmus <- readRDS(paste0(path, 'Mmus.non_neurons.iter_cluster_annotated.rds'))
Mmus <- subset(Mmus, cells = rownames(Mmus@meta.data)[which(Mmus@meta.data$`Refined family` %in% 
    c('Ependymal cells', 'Astrocytes', 'Oligodendrocytes', 'Oligodendrocyte precursor cells',
     'Vascular cells', 'Microglia', 'Fibroblasts'))])
Mmus_metagene <- get_metagene_obj(Mmus, orthogroups, "Mmus")
rm(Mmus)
saveRDS(Mmus_metagene, "Mmus.glia.metagene.rds")

In [14]:
Hsap <- readRDS(paste0(path, 'Hsap.non_neurons.iter_cluster_annotated.rds'))
Hsap <- subset(Hsap, cells = rownames(Hsap@meta.data)[which(Hsap@meta.data$`Refined family` %in% 
    c('Ependymal cells', 'Astrocytes', 'Oligodendrocytes', 'Oligodendrocyte precursor cells',
     'Vascular cells', 'Microglia', 'Fibroblasts'))])
Hsap_metagene <- get_metagene_obj(Hsap, orthogroups, "Hsap")
rm(Hsap)
saveRDS(Hsap_metagene, "Hsap.glia.metagene.rds")

In [ ]:
Bflo <- readRDS('/mnt/data01/yuanzhen/03.sensory_neurosecretory_evo/01.data/03.atlas/Bflo/brain_integrated/4000_40_30/RNA_harmony_integrated.RDS')
Bflo <- subset(Bflo, idents = c(8,12,4,5,20,6,3, 22, 19, 18)) # select glia population of amphioxus
Bflo@meta.data[,'Refined subtype'] <- Bflo@meta.data$`RNA_snn_res.1`
Bflo@meta.data[,'Refined family'] <- Bflo@meta.data$`RNA_snn_res.1`
Bflo@meta.data$DonorID <- Bflo@meta.data$orig.ident
Bflo@meta.data$Species <- 'Bflo'
Bflo_metagene <- get_metagene_obj(Bflo, orthogroups, "Bflo")
rm(Bflo)
saveRDS(Bflo_metagene, "Bflo.glia.metagene.rds")

In [12]:
write.table(orthogroups, file = "orthogroups.5chordates.txt", sep = '\t', quote = F, 
            row.names = F, col.names = T)